In [0]:

from pyspark.sql import functions as F

silver_table = "job_market.silver.hacker_news_api_posts"

df = spark.readStream.table(
    "job_market.silver.hacker_news_api_posts"
)

# Technologien und ihre Suchmuster
technology_patterns = {
    "Python": r"(?i)\bpython\b",
    "SQL": r"(?i)\bsql\b",
    "Spark": r"(?i)\b(?:apache\s+spark|spark)\b",
    "PySpark": r"(?i)\bpyspark\b",
    "Airflow": r"(?i)\bairflow\b",
    "Databricks": r"(?i)\bdatabricks\b",
    "Kafka": r"(?i)\bkafka\b",
    "Docker": r"(?i)\bdocker\b",
    "Kubernetes": r"(?i)\b(?:kubernetes|k8s)\b",
    "AWS": r"(?i)\baws\b",
    "Azure": r"(?i)\bazure\b",
    "GCP": r"(?i)\b(?:gcp|google cloud)\b",
    "Snowflake": r"(?i)\bsnowflake\b",
    "PostgreSQL": r"(?i)\bpostgres(?:ql)?\b",
    "dbt": r"(?i)\bdbt\b"
}

# Für jede Technologie prüfen, ob sie im Text vorkommt
skill_columns = [
    F.when(
        F.col("text_plain").rlike(pattern),
        F.lit(technology)
    )
    for technology, pattern in technology_patterns.items()
]

df_enriched = df.withColumn(
    "technologies_mentioned",
    F.filter(
        F.array(*skill_columns),
        lambda x: x.isNotNull()
    )
)


In [0]:

role_patterns = {
    "Data Engineer": r"(?i)\bdata engineers?\b",
    "Data Scientist": r"(?i)\bdata scientists?\b",
    "Data Analyst": r"(?i)\bdata analysts?\b",
    "ML Engineer": r"(?i)\b(?:machine learning|ml) engineers?\b",
    "Software Engineer": r"(?i)\b(?:software|backend|frontend|fullstack|full-stack) engineers?\b",
    "DevOps / SRE": r"(?i)\bdevops\b|\bsite reliability engineers?\b",
    "Security Engineer": r"(?i)\bsecurity engineers?\b",
    "Product Manager": r"(?i)\bproduct managers?\b"
}

role_columns = [
    F.when(
        F.col("text_plain").rlike(pattern),
        F.lit(role)
    )
    for role, pattern in role_patterns.items()
]

df_enriched = df_enriched.withColumn(
    "role_families_mentioned",
    F.filter(
        F.array(*role_columns),
        lambda x: x.isNotNull()
    )
)


In [0]:

header = F.lower(
    F.substring(F.col("text_plain"), 1, 300)
)

df_enriched = df_enriched.withColumn(
    "work_mode_candidate",
    F.when(
        header.rlike(r"\bhybrid\b"),
        "HYBRID"
    )
    .when(
        header.rlike(r"\bremote\b") &
        header.rlike(r"\bon[\s-]?site\b"),
        "MIXED_OR_AMBIGUOUS"
    )
    .when(
        header.rlike(r"\bremote\b"),
        "REMOTE"
    )
    .when(
        header.rlike(r"\bon[\s-]?site\b"),
        "ONSITE"
    )
    .otherwise("UNKNOWN")
)


In [0]:

df_enriched = (
    df_enriched
    .withColumn(
        "skills",
        F.col("technologies_mentioned")
    )
    .withColumn(
        "is_remote",
        F.when(
            F.col("work_mode_candidate").isin(
                "REMOTE", "HYBRID", "MIXED_OR_AMBIGUOUS"
            ),
            F.lit(True)
        )
        .when(
            F.col("work_mode_candidate") == "ONSITE",
            F.lit(False)
        )
        .otherwise(F.lit(None).cast("boolean"))
    )
)


In [0]:

target_table = "job_market.silver.hacker_news_api_enriched_v2"

checkpoint_path = (
    "/Volumes/job_market/landing/raw_files/"
    "_pipeline_state/hn_api_enriched_v2"
)

query = (
    df_enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()


In [0]:
%sql

SELECT
    work_mode_candidate,
    COUNT(*) AS total_posts
FROM job_market.silver.hacker_news_api_enriched_v2
GROUP BY work_mode_candidate
ORDER BY total_posts DESC;


In [0]:
%sql

SELECT
    role.role_family,
    COUNT(DISTINCT post_id) AS total_posts
FROM job_market.silver.hacker_news_api_enriched_v2,
LATERAL EXPLODE(role_families_mentioned)
    AS role(role_family)
GROUP BY role.role_family
ORDER BY total_posts DESC;


In [0]:
%sql

SELECT
    tech.skill AS technology,
    COUNT(DISTINCT post_id) AS total_posts
FROM job_market.silver.hacker_news_api_enriched_v2,
LATERAL EXPLODE(technologies_mentioned) AS tech(skill)
GROUP BY tech.skill
ORDER BY total_posts DESC;


In [0]:
%sql

SELECT
    COUNT(*) AS total_posts,
    COUNT(DISTINCT post_id) AS unique_posts,
    COUNT_IF(is_remote = true) AS remote_posts,
    COUNT_IF(is_remote = false) AS onsite_posts,
    COUNT_IF(is_remote IS NULL) AS unknown_posts
FROM job_market.silver.hacker_news_api_enriched_v2;
